# K0 Final Pipeline — 逐行中文注释复现版

这个 notebook 只保留复现 K0 最终提交所需的可运行部分。

保留内容：
- 读取 Kaggle 数据集
- 重建 K0 特征处理流程
- 加载并恢复保存的 RNG state
- 使用 SMOTE 平衡训练集
- 训练最终 XGBoost 模型
- 与参考 submission 逐行比较，验证是否完全复现


In [ ]:
# Core imports
# 导入 Path，用来更方便地处理文件路径。
from pathlib import Path
# 导入 sys，用来查看 Python 版本等系统信息。
import sys
# 导入 platform，用来输出当前运行平台信息。
import platform
# 导入 zipfile，用来解压包含 RNG state 和 submission 的资产包。
import zipfile
# 导入 pickle，用来读取保存好的 NumPy RNG state。
import pickle

# 导入 NumPy，主要用于恢复随机数生成器状态。
import numpy as np
# 导入 pandas，用来读取和处理表格数据。
import pandas as pd

# 导入缺失值填补器，用来补全剩余缺失值。
from sklearn.impute import SimpleImputer
# 导入 OneHotEncoder，用来把类别变量转成数值特征。
from sklearn.preprocessing import OneHotEncoder
# 导入 shuffle，用来按保存的 RNG state 重放训练集打乱。
from sklearn.utils import shuffle

# 导入 SMOTE，用来平衡 True / False 类别。
from imblearn.over_sampling import SMOTE
# 导入 XGBoost，最终主模型使用 XGBClassifier。
import xgboost as xgb

# 让 pandas 显示全部列，方便检查特征表。
pd.set_option("display.max_columns", None)

# 设置输出目录；Kaggle 环境用 /kaggle/working，本地环境用当前目录。
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
# 确保输出目录存在，不存在就创建。
OUT.mkdir(parents=True, exist_ok=True)

# 打印 Python 版本，方便记录运行环境。
print("Python:", sys.version.split()[0])
# 打印平台信息，方便记录运行环境。
print("Platform:", platform.platform())

Python: 3.7.12
Platform: Linux-6.6.122+-x86_64-with-debian-bullseye-sid


## 1. 定位数据集和复现资产文件

这个 notebook 需要两类输入：

1. Kaggle Spaceship Titanic 数据：`train.csv`、`test.csv`、`sample_submission.csv`
2. 复现资产文件：保存好的 RNG state 和参考 submission

在 Kaggle 上运行时，需要把复现用的 zip 文件作为 input dataset 加进来。

代码会自动搜索普通命名、带时间戳命名，以及旧版 results.zip 里的文件。


In [ ]:
# 定义函数：在多个目录中查找指定文件名。
def find_file(filename, roots):
    # 列表中的一个特征名或文件名前缀。
    """在多个目录里按完整文件名搜索。"""
    # 遍历所有候选搜索目录。
    for root in roots:
        # 把目录统一转换成 Path 对象，方便后续路径操作。
        root = Path(root)
        # 如果这个候选目录不存在，就跳过。
        if not root.exists():
            # 执行这一行，保持原 pipeline 的操作顺序。
            continue
        # 先检查当前目录下是否直接存在目标文件。
        direct = root / filename
        # 如果直接拼出的文件路径存在，就直接返回。
        if direct.exists():
            # 返回直接找到的文件路径。
            return direct
        # 递归搜索当前目录下符合文件名的文件。
        for p in root.rglob(filename):
            # 返回递归搜索找到的文件路径。
            return p
    # 如果所有目录都没找到，就返回 None。
    return None

# 定义函数：按文件名前缀和后缀查找文件，支持带时间戳的文件名。
def find_first_by_prefix(prefix, suffix, roots):
    # 列表中的一个特征名或文件名前缀。
    """按前缀/后缀搜索，例如 rng_state_before_shuffle_*.pkl。"""
    # 遍历所有候选搜索目录。
    for root in roots:
        # 把目录统一转换成 Path 对象，方便后续路径操作。
        root = Path(root)
        # 如果这个候选目录不存在，就跳过。
        if not root.exists():
            # 执行这一行，保持原 pipeline 的操作顺序。
            continue
        # 查找所有符合前缀和后缀的文件，并排序保证结果稳定。
        matches = sorted(root.rglob(f"{prefix}*{suffix}"))
        # 如果找到了匹配文件，就返回第一个匹配项。
        if matches:
            # 返回排序后的第一个匹配文件。
            return matches[0]
    # 如果所有目录都没找到，就返回 None。
    return None

# 定义函数：如果 input 里有复现资产 zip，就自动解压。
def extract_reproduction_zip_if_needed(roots, extract_dir):
    # 列表中的一个特征名或文件名前缀。
    """
    # 执行这一行，保持原 pipeline 的操作顺序。
    解压 run-07 资产包。
    # 执行这一行，保持原 pipeline 的操作顺序。
    支持两种格式：
    # 执行这一行，保持原 pipeline 的操作顺序。
    1. 简洁命名：rng_state_before_shuffle.pkl
    # 执行这一行，保持原 pipeline 的操作顺序。
    2. 时间戳命名：rng_state_before_shuffle_1779954951.pkl
    # 列表中的一个特征名或文件名前缀。
    """
    # 创建解压目录，确保后面可以把 zip 解压进去。
    extract_dir.mkdir(parents=True, exist_ok=True)

    # 开始定义一个列表。
    wanted_prefixes = [
        # 列表中的一个特征名或文件名前缀。
        "rng_state_before_shuffle",
        # 列表中的一个特征名或文件名前缀。
        "rng_state_before_smote",
        # 列表中的一个特征名或文件名前缀。
        "rng_state_before_final_model",
        # 列表中的一个特征名或文件名前缀。
        "k0_run_07_reference_submission",
        # 列表中的一个特征名或文件名前缀。
        "k0_random_run_07_clean_submission",
        # 列表中的一个特征名或文件名前缀。
        "k0_random_run_11",
    # 结束当前的多行结构。
    ]

    # 遍历所有候选搜索目录。
    for root in roots:
        # 把目录统一转换成 Path 对象，方便后续路径操作。
        root = Path(root)
        # 如果这个候选目录不存在，就跳过。
        if not root.exists():
            # 执行这一行，保持原 pipeline 的操作顺序。
            continue
        # 递归搜索当前目录下的 zip 文件。
        for zp in root.rglob("*.zip"):
            # 尝试执行下面的代码；如果版本不兼容，再进入 except 分支。
            try:
                # 打开 zip 文件，准备检查里面的文件名。
                with zipfile.ZipFile(zp, "r") as z:
                    # 只取 zip 内部文件的文件名，忽略目录层级。
                    basenames = [Path(name).name for name in z.namelist()]
                    # 判断这个 zip 里是否包含需要的 RNG state 或参考 submission 文件。
                    if any(any(base.startswith(prefix) for prefix in wanted_prefixes) for base in basenames):
                        # 把匹配到的资产 zip 解压到指定目录。
                        z.extractall(extract_dir)
                        # 打印当前成功解压的 zip 文件路径。
                        print("Extracted assets from:", zp)
                        # 返回解压后的目录。
                        return extract_dir
            # 如果某个 zip 读取失败，捕获错误并继续搜索其他 zip。
            except Exception as e:
                # 打印被跳过的 zip 文件和原因。
                print("Skipped zip:", zp, e)
    # 如果所有目录都没找到，就返回 None。
    return None

# 开始定义一个列表。
SEARCH_ROOTS = [
    # Kaggle input 数据集目录。
    Path("/kaggle/input"),
    # 当前工作目录。
    Path("."),
    # ChatGPT / 本地沙盒中常见的数据目录。
    Path("/mnt/data"),
# 结束当前的多行结构。
]

# 先尝试解压 zip，再统一搜索。
# 设置复现资产 zip 的解压目录。
extract_dir = OUT / "run07_assets_extracted"
# 自动尝试解压包含 RNG state 和 submission 的 zip。
extract_reproduction_zip_if_needed(SEARCH_ROOTS, extract_dir)
# 把原始搜索目录、输出目录和解压目录合并成总搜索范围。
ALL_ROOTS = SEARCH_ROOTS + [OUT, extract_dir]

# 创建字典，用来保存三个 RNG state 文件路径。
asset_paths = {}
# 开始定义一个字典。
state_specs = {
    # shuffle 前的 RNG state，对应训练集打乱前的随机状态。
    "before_shuffle": "rng_state_before_shuffle",
    # SMOTE 前的 RNG state，对应合成少数类样本前的随机状态。
    "before_smote": "rng_state_before_smote",
    # final XGBoost 训练前的 RNG state。
    "before_final_model": "rng_state_before_final_model",
# 结束当前的多行结构。
}

# 依次查找三个关键 RNG state 文件。
for key, prefix in state_specs.items():
    # 优先找简洁命名；找不到就接受带时间戳的命名。
    # 优先查找没有时间戳的标准命名文件。
    path = find_file(f"{prefix}.pkl", ALL_ROOTS)
    # 如果没有找到简洁命名文件，就继续尝试其他命名格式。
    if path is None:
        # 如果标准命名没找到，就查找带时间戳的文件。
        path = find_first_by_prefix(prefix + "_", ".pkl", ALL_ROOTS)
    # 如果缺少这个 RNG state 文件，就立刻报错停止。
    assert path is not None, f"Missing RNG state file for {key}"
    # 把找到的 RNG state 路径保存到字典里。
    asset_paths[key] = path

# reference submission：优先 clean reference，其次 run-07 clean submission，其次旧 run-11 submission。
# 优先查找标准命名的 reference submission。
reference_path = find_file("k0_run_07_reference_submission.csv", ALL_ROOTS)
# 如果当前格式没找到 reference submission，就尝试下一种格式。
if reference_path is None:
    # 查找带时间戳或旧命名的参考 submission。
    reference_path = find_first_by_prefix("k0_random_run_07_clean_submission_", ".csv", ALL_ROOTS)
# 如果当前格式没找到 reference submission，就尝试下一种格式。
if reference_path is None:
    # 查找带时间戳或旧命名的参考 submission。
    reference_path = find_first_by_prefix("k0_random_run_11_", ".csv", ALL_ROOTS)
# 如果找不到参考 submission，就停止运行。
assert reference_path is not None, "Missing reference submission file"

# 打印标题，下面列出找到的 RNG state 文件。
print("RNG state files:")
# 逐个打印已经找到的 RNG state 文件路径。
for key, path in asset_paths.items():
    # 打印每个 RNG state 的名称和具体路径。
    print(f"  {key}: {path}")

# 打印参考 submission 的路径。
print("Reference submission:", reference_path)

RNG state files:
  before_shuffle: /kaggle/input/datasets/zhumengfighting/k0-reproduction-assets-clean/rng_state_before_shuffle.pkl
  before_smote: /kaggle/input/datasets/zhumengfighting/k0-reproduction-assets-clean/rng_state_before_smote.pkl
  before_final_model: /kaggle/input/datasets/zhumengfighting/k0-reproduction-assets-clean/rng_state_before_final_model.pkl
Reference submission: /kaggle/input/datasets/zhumengfighting/k0-reproduction-assets-clean/k0_run_07_reference_submission.csv


In [3]:
# 定义函数：自动定位 train/test/sample_submission 三个 Kaggle 数据文件。
def find_dataset_root():
    # 开始定义一个列表。
    candidates = [
        # Kaggle input 数据集目录。
        Path("/kaggle/input/spaceship-titanic"),
        # Kaggle input 数据集目录。
        Path("/kaggle/input"),
        # 当前工作目录。
        Path("."),
        # ChatGPT / 本地沙盒中常见的数据目录。
        Path("/mnt/data"),
    # 结束当前的多行结构。
    ]
    # 查找训练集 train.csv 的路径。
    train_path = find_file("train.csv", candidates)
    # 查找测试集 test.csv 的路径。
    test_path = find_file("test.csv", candidates)
    # 查找 sample_submission.csv 的路径。
    sample_path = find_file("sample_submission.csv", candidates)

    # 确认 train.csv 找到了，否则停止。
    assert train_path is not None, "train.csv not found"
    # 确认 test.csv 找到了，否则停止。
    assert test_path is not None, "test.csv not found"
    # 确认 sample_submission.csv 找到了，否则停止。
    assert sample_path is not None, "sample_submission.csv not found"

    # 返回三个数据文件路径。
    return train_path, test_path, sample_path

# 调用函数，得到三个数据文件路径。
train_path, test_path, sample_path = find_dataset_root()

# 读取原始训练集。
train_raw = pd.read_csv(train_path)
# 读取原始测试集。
test_raw = pd.read_csv(test_path)
# 读取提交模板，用来生成最终 submission。
sample_submission = pd.read_csv(sample_path)

# 打印训练集大小和路径，确认读取正确。
print("Train:", train_raw.shape, train_path)
# 打印测试集大小和路径，确认读取正确。
print("Test:", test_raw.shape, test_path)
# 打印提交模板大小和路径，确认读取正确。
print("Sample submission:", sample_submission.shape, sample_path)

Train: (8693, 14) /kaggle/input/spaceship-titanic/train.csv
Test: (4277, 13) /kaggle/input/spaceship-titanic/test.csv
Sample submission: (4277, 2) /kaggle/input/spaceship-titanic/sample_submission.csv


## 2. K0 特征重建与预处理

这一部分负责把原始 train/test 转成最终 K0 pipeline 使用的特征矩阵。

核心步骤：
- 合并 train 和 test，只处理非标签特征
- 根据 CryoSleep 和消费的物理关系补值
- 计算五项消费总和 Expenses
- 用 PassengerId 前四位构造 Room / group 信息
- 用同组乘客信息进行 RoomFill
- 拆分 Cabin 为 deck / number / side
- 对剩余缺失值做统计补全
- 对类别特征做 One-Hot 编码

这里的处理顺序保持和原 K0 复现流程一致。


In [ ]:
# 定义函数：创建返回 dense array 的 OneHotEncoder。
def one_hot_encoder_dense():
    # scikit-learn renamed `sparse` to `sparse_output` in newer versions.
    # 尝试执行下面的代码；如果版本不兼容，再进入 except 分支。
    try:
        # 返回兼容当前 sklearn 版本的 OneHotEncoder。
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    # 如果当前 sklearn 版本不支持新参数，就使用旧参数写法。
    except TypeError:
        # 返回兼容当前 sklearn 版本的 OneHotEncoder。
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# 定义函数：把原始 train/test 转成 K0 pipeline 的特征矩阵。
def build_k0_features(train_df, test_df, use_roomfill=True):
    # 把 train 和 test 上下合并，只为了统一处理特征列。
    train_test = pd.concat([train_df, test_df], axis=0, sort=False).copy()

    # 定义五个消费字段。
    expense_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

    # CryoSleep=True means the passenger cannot spend on paid services.
    # Kept in the same operation order as the original K0 replay notebook.
    # 根据 CryoSleep 规则修正五项消费字段。
    train_test.loc[:, expense_cols] = train_test.apply(
        # 如果乘客处于 CryoSleep，就把消费强制设为 0。
        lambda row: 0 if row.CryoSleep == True else row,
        # 按行执行 apply。
        axis=1
    # 结束当前的多行结构。
    )

    # Expenses: total spending across the five service columns.
    # 把五项消费加总成 Expenses 总消费特征。
    train_test["Expenses"] = train_test.loc[:, expense_cols].sum(axis=1)

    # If all spending is zero and CryoSleep is missing, infer CryoSleep=True.
    # 根据总消费反推缺失的 CryoSleep。
    train_test.loc[:, ["CryoSleep"]] = train_test.apply(
        # 如果总消费为 0 且 CryoSleep 缺失，就推断 CryoSleep=True。
        lambda row: True if row.Expenses == 0 and pd.isna(row.CryoSleep) else row,
        # 按行执行 apply。
        axis=1
    # 结束当前的多行结构。
    )

    # Room/group prefix from PassengerId.
    # 从 PassengerId 前四位提取 Room / group 前缀。
    train_test.loc[:, ["Room"]] = train_test.PassengerId.apply(lambda x: x[0:4])

    # 如果启用 RoomFill，就用同组乘客信息补缺失。
    if use_roomfill:
        # Build group-level lookup tables for non-label passenger information.
        # 建立 Room 到 VIP 的查找表，只保留非缺失值。
        guide_vip = train_test.loc[:, ["Room", "VIP"]].dropna().drop_duplicates("Room")
        # 建立 Room 到 Cabin 的查找表，只保留非缺失值。
        guide_cabin = train_test.loc[:, ["Room", "Cabin"]].dropna().drop_duplicates("Room")
        # 建立 Room 到 HomePlanet 的查找表，只保留非缺失值。
        guide_homeplanet = train_test.loc[:, ["Room", "HomePlanet"]].dropna().drop_duplicates("Room")
        # 建立 Room 到 Destination 的查找表，只保留非缺失值。
        guide_destination = train_test.loc[:, ["Room", "Destination"]].dropna().drop_duplicates("Room")

        # 把 Cabin 查找表合并回 train_test，生成辅助列 Cabin_y。
        train_test = pd.merge(train_test, guide_cabin, how="left", on="Room", suffixes=("", "_y"))
        # 把 VIP 查找表合并回 train_test，生成辅助列 VIP_y。
        train_test = pd.merge(train_test, guide_vip, how="left", on="Room", suffixes=("", "_y"))
        # 把 HomePlanet 查找表合并回 train_test，生成辅助列 HomePlanet_y。
        train_test = pd.merge(train_test, guide_homeplanet, how="left", on="Room", suffixes=("", "_y"))
        # 把 Destination 查找表合并回 train_test，生成辅助列 Destination_y。
        train_test = pd.merge(train_test, guide_destination, how="left", on="Room", suffixes=("", "_y"))

        # 开始补全 VIP 字段。
        train_test.loc[:, ["VIP"]] = train_test.apply(
            # 如果 VIP 原值缺失，就使用同组乘客提供的 VIP_y。
            lambda row: row.VIP_y if pd.isna(row.VIP) else row,
            # 按行执行 apply。
            axis=1
        # 结束当前的多行结构。
        )
        # 开始补全 Cabin 字段。
        train_test.loc[:, ["Cabin"]] = train_test.apply(
            # 如果 Cabin 原值缺失，就使用同组乘客提供的 Cabin_y。
            lambda row: row.Cabin_y if pd.isna(row.Cabin) else row,
            # 按行执行 apply。
            axis=1
        # 结束当前的多行结构。
        )
        # 开始补全 HomePlanet 字段。
        train_test.loc[:, ["HomePlanet"]] = train_test.apply(
            # 如果 HomePlanet 原值缺失，就使用同组乘客提供的 HomePlanet_y。
            lambda row: row.HomePlanet_y if pd.isna(row.HomePlanet) else row,
            # 按行执行 apply。
            axis=1
        # 结束当前的多行结构。
        )
        # 开始补全 Destination 字段。
        train_test.loc[:, ["Destination"]] = train_test.apply(
            # 如果 Destination 原值缺失，就使用同组乘客提供的 Destination_y。
            lambda row: row.Destination_y if pd.isna(row.Destination) else row,
            # 按行执行 apply。
            axis=1
        # 结束当前的多行结构。
        )

    # Split Cabin into deck / number / side.
    # 从 Cabin 中拆出第一段：Deck。
    train_test.loc[:, ["Cabin_1"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 0]  # Deck
    # 从 Cabin 中拆出第二段：Cabin number；后面不作为核心特征。
    train_test.loc[:, ["Cabin_2"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 1]  # Number, not used later
    # 从 Cabin 中拆出第三段：Side。
    train_test.loc[:, ["Cabin_3"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 2]  # Side

    # 定义数值特征列。
    num_cols = ["ShoppingMall", "FoodCourt", "RoomService", "Spa", "VRDeck", "Expenses", "Age"]
    # 定义类别特征列。
    cat_cols = ["CryoSleep", "Cabin_1", "Cabin_3", "VIP", "HomePlanet", "Destination"]

    # 只保留模型需要的数值列、类别列和标签列。
    train_test = train_test[num_cols + cat_cols + ["Transported"]].copy()

    # 创建数值缺失值填补器，使用均值补全。
    num_imp = SimpleImputer(strategy="mean")
    # 创建类别缺失值填补器，使用众数补全。
    cat_imp = SimpleImputer(strategy="most_frequent")
    # 创建 One-Hot 编码器。
    ohe = one_hot_encoder_dense()

    # 对数值列进行缺失值填补。
    train_test[num_cols] = pd.DataFrame(
        # 创建数值缺失值填补器，使用均值补全。
        num_imp.fit_transform(train_test[num_cols]),
        # 给补全后的数值矩阵恢复原列名。
        columns=num_cols
    # 结束当前的多行结构。
    )

    # 对类别列进行缺失值填补。
    train_test[cat_cols] = pd.DataFrame(
        # 创建类别缺失值填补器，使用众数补全。
        cat_imp.fit_transform(train_test[cat_cols]),
        # 给补全后的类别矩阵恢复原列名。
        columns=cat_cols
    # 结束当前的多行结构。
    )

    # 对类别特征进行 One-Hot 编码。
    encoded = pd.DataFrame(
        # 拟合 One-Hot 编码器，并把类别列转成 0/1 特征。
        ohe.fit_transform(train_test[cat_cols]),
        # 使用 One-Hot 编码后的新特征名作为列名。
        columns=ohe.get_feature_names_out()
    # 结束当前的多行结构。
    )

    # 删除原始类别列，避免和 One-Hot 列重复。
    train_test = train_test.drop(columns=cat_cols)
    # 把 train 和 test 上下合并，只为了统一处理特征列。
    train_test = pd.concat([train_test, encoded], axis=1)

    # 切回训练集部分：Transported 非空的行。
    train_processed = train_test[train_test["Transported"].notna()].copy()
    # 切回测试集部分：Transported 为空的行，并删除标签列。
    test_processed = train_test[train_test["Transported"].isna()].drop(columns=["Transported"]).copy()

    # 切回训练集部分：Transported 非空的行。
    train_processed["Transported"] = train_processed["Transported"].astype(int)

    # 从训练集里删除标签列，得到训练特征 X。
    X = train_processed.drop(columns=["Transported"])
    # 取出 Transported 作为训练标签 y。
    y = train_processed["Transported"]

    # 返回训练特征、训练标签和测试特征。
    return X, y, test_processed

# 执行 K0 特征构建流程，生成模型输入。
X, y, test_features = build_k0_features(train_raw, test_raw, use_roomfill=True)

# 打印训练特征矩阵的形状。
print("X:", X.shape)
# 打印训练标签的形状。
print("y:", y.shape)
# 打印测试特征矩阵的形状。
print("test_features:", test_features.shape)
# 打印 SMOTE 前的类别分布。
print("Class balance before SMOTE:")
# 输出训练标签中 True / False 的数量。
print(y.value_counts())

X: (8693, 28)
y: (8693,)
test_features: (4277, 28)
Class balance before SMOTE:
1    4378
0    4315
Name: Transported, dtype: int64


## 3. 恢复保存的 RNG states

K0 pipeline 中有几个会消耗随机数的步骤：

1. shuffle 训练行顺序
2. SMOTE 选择邻居并合成样本
3. XGBoost 训练中的随机采样

为了让复现结果完全一致，需要在这些关键随机步骤之前恢复保存好的 NumPy RNG state。


In [ ]:
# 定义函数：读取保存的 RNG state，并恢复 NumPy 随机数状态。
def load_numpy_rng_state(path):
    # 以二进制方式打开保存的 pkl 文件。
    with open(path, "rb") as f:
        # 从 pkl 文件中读取保存好的 NumPy RNG state。
        state = pickle.load(f)
    # 把 NumPy 随机数生成器恢复到保存时的状态。
    np.random.set_state(state)
    # 打印已加载的 RNG state 文件路径。
    print("Loaded RNG state:", path)

# 1) Replay data shuffle.
# 在 shuffle 前恢复对应的 RNG state。
load_numpy_rng_state(asset_paths["before_shuffle"])
# 用恢复后的随机状态打乱 X 和 y，保证顺序与参考运行一致。
X, y = shuffle(X, y)
# 重置 X 的索引，避免打乱后保留旧索引。
X = X.reset_index(drop=True)
# 重置 y 的索引，保证和 X 对齐。
y = y.reset_index(drop=True)

# Remove low-value / redundant columns selected out in the K0 pipeline.
# 开始定义一个列表。
drop_list = [
    # 列表中的一个特征名或文件名前缀。
    "ShoppingMall",
    # 列表中的一个特征名或文件名前缀。
    "Age",
    # 列表中的一个特征名或文件名前缀。
    "CryoSleep_True",
    # 列表中的一个特征名或文件名前缀。
    "HomePlanet_Earth",
    # 列表中的一个特征名或文件名前缀。
    "HomePlanet_Europa",
    # 列表中的一个特征名或文件名前缀。
    "VIP_True",
    # 列表中的一个特征名或文件名前缀。
    "HomePlanet_Mars",
    # 列表中的一个特征名或文件名前缀。
    "Destination_PSO J318.5-22",
    # 列表中的一个特征名或文件名前缀。
    "VIP_False",
    # 列表中的一个特征名或文件名前缀。
    "Destination_55 Cancri e",
    # 列表中的一个特征名或文件名前缀。
    "FoodCourt",
    # 列表中的一个特征名或文件名前缀。
    "Destination_TRAPPIST-1e",
# 结束当前的多行结构。
]

# 从训练特征中删除这些低价值或冗余列。
X = X.drop(columns=drop_list)
# 从测试特征中删除同样的列，保证 train/test 特征空间一致。
test_features = test_features.drop(columns=drop_list)

# 打印提示：下面是特征筛选后的形状。
print("After feature pruning:")
# 打印训练特征矩阵的形状。
print("X:", X.shape)
# 打印测试特征矩阵的形状。
print("test_features:", test_features.shape)

Loaded RNG state: /kaggle/input/datasets/zhumengfighting/k0-reproduction-assets-clean/rng_state_before_shuffle.pkl
After feature pruning:
X: (8693, 16)
test_features: (4277, 16)


In [ ]:
# 2) Replay SMOTE.
# 在执行 SMOTE 前恢复对应的 RNG state。
load_numpy_rng_state(asset_paths["before_smote"])

# 尝试执行下面的代码；如果版本不兼容，再进入 except 分支。
try:
    # 创建 SMOTE 对象，把少数类过采样到和多数类相同数量。
    smote = SMOTE(sampling_strategy=1, n_jobs=-1)
# 如果当前 sklearn 版本不支持新参数，就使用旧参数写法。
except TypeError:
    # 创建 SMOTE 对象，把少数类过采样到和多数类相同数量。
    smote = SMOTE(sampling_strategy=1)

# 执行 SMOTE，生成平衡后的训练特征和标签。
X_sm, y_sm = smote.fit_resample(X, y)
# 用 SMOTE 后的特征替换原训练特征。
X = X_sm
# 用 SMOTE 后的标签替换原训练标签。
y = y_sm

# 打印 SMOTE 后的类别分布。
print("Class balance after SMOTE:")
# 输出平衡后的 True / False 数量。
print(pd.Series(y).value_counts())

Loaded RNG state: /kaggle/input/datasets/zhumengfighting/k0-reproduction-assets-clean/rng_state_before_smote.pkl
Class balance after SMOTE:
1    4378
0    4378
Name: Transported, dtype: int64


## 4. 最终 XGBoost 模型与逐行复现检查

这一部分会：

1. 设置最终 XGBoost 参数
2. 恢复 final model 训练前的 RNG state
3. 训练模型并预测 test
4. 生成新的 reproduced submission
5. 读取参考 submission
6. 逐行比较 4,277 个预测是否完全一致

如果 `Different rows = 0`，说明复现成功。


In [7]:
# 开始定义一个字典。
xgb_params_final = {
    # 列表中的一个特征名或文件名前缀。
    "lambda": 3.0610042624477543,
    # 列表中的一个特征名或文件名前缀。
    "alpha": 4.581902571574289,
    # 列表中的一个特征名或文件名前缀。
    "colsample_bytree": 0.9241969052729379,
    # 列表中的一个特征名或文件名前缀。
    "subsample": 0.9527591724824661,
    # 列表中的一个特征名或文件名前缀。
    "learning_rate": 0.06672065863100594,
    # 列表中的一个特征名或文件名前缀。
    "n_estimators": 730,
    # 列表中的一个特征名或文件名前缀。
    "max_depth": 5,
    # 列表中的一个特征名或文件名前缀。
    "min_child_weight": 1,
    # 列表中的一个特征名或文件名前缀。
    "num_parallel_tree": 1,
# 结束当前的多行结构。
}

# 在最终 XGBoost 训练前恢复对应的 RNG state。
load_numpy_rng_state(asset_paths["before_final_model"])

# 用最终参数创建 XGBoost 分类器。
final_model = xgb.XGBClassifier(**xgb_params_final)
# 训练最终模型，并对测试集生成预测结果。
pred = final_model.fit(X, y).predict(test_features)

# 复制提交模板，准备填入预测结果。
submission = sample_submission.copy()
# 把模型预测结果写入 Transported 列。
submission["Transported"] = pred
# 把预测值转换成布尔值 True / False，符合 Kaggle 提交格式。
submission["Transported"] = submission["Transported"] > 0.5

# 设置新生成 submission 的保存路径。
reproduced_path = OUT / "k0_run_13_reproduced_submission.csv"
# 把复现得到的 submission 保存成 csv 文件。
submission.to_csv(reproduced_path, index=False)

# 读取参考 submission，用来逐行比较。
reference = pd.read_csv(reference_path)
# 计算新 submission 和参考 submission 完全相同的比例。
same_ratio = (reference["Transported"] == submission["Transported"]).mean()
# 计算新 submission 和参考 submission 不同的行数。
different_rows = int((reference["Transported"] != submission["Transported"]).sum())

# 打印新 submission 的保存位置。
print("Saved reproduced submission:", reproduced_path)
# 打印相同比例；1.0 表示全部一致。
print("Same ratio:", same_ratio)
# 打印不同预测的行数；0 表示完全一致。
print("Different rows:", different_rows)

# 执行这一行，保持原 pipeline 的操作顺序。
print("\\nReference distribution:")
# 输出参考 submission 中 True / False 的数量。
print(reference["Transported"].value_counts())

# 执行这一行，保持原 pipeline 的操作顺序。
print("\\nReproduced distribution:")
# 输出复现 submission 中 True / False 的数量。
print(submission["Transported"].value_counts())

# 如果有任何一行不同，就报错，说明复现失败。
assert different_rows == 0, "Reproduction failed: the new submission differs from the reference submission."
# 执行这一行，保持原 pipeline 的操作顺序。
print("\\nExact reproduction successful.")

Loaded RNG state: /kaggle/input/datasets/zhumengfighting/k0-reproduction-assets-clean/rng_state_before_final_model.pkl
Saved reproduced submission: /kaggle/working/k0_run_13_reproduced_submission.csv
Same ratio: 1.0
Different rows: 0
\nReference distribution:
True     2282
False    1995
Name: Transported, dtype: int64
\nReproduced distribution:
True     2282
False    1995
Name: Transported, dtype: int64
\nExact reproduction successful.


## Demo 核心结论

- 最终 pipeline 的核心是 RoomFill + SMOTE + XGBoost。
- notebook 在关键随机步骤前恢复保存的 RNG state。
- 新生成的 submission 与参考 submission 逐行完全一致时，说明不仅分数复现，提交文件本身也复现。
